# ETL-21 · dim_localidade e dim_via

- **Etapa:** `07c_dim_localidade_via` (ETL-21)
- **Responsável:** Guilherme Calistrato (`@guicalistrato`)
- **Revisor:** Davi Mello (`@davi01mello`)
- **Issue:** #23 (ETL-21)
- **Entradas:** `data/intermediarios/03_texto.parquet` (`dim_localidade`, saída da ETL-14) e `data/intermediarios/05_nulos_categorias.parquet` (`dim_via`, saída da ETL-16) *(na execução encadeada final, ambas apontam para `06_limpo.parquet`)*
- **Saídas:** `data/intermediarios/dims/dim_localidade.parquet` e `data/intermediarios/dims/dim_via.parquet`

Cria as dimensões `dim_localidade` e `dim_via` do modelo estrela, com chaves substitutas sequenciais `sk_*`, ordenação determinística e os membros especiais `-1` (`NÃO INFORMADO`) e `-2` (`NÃO SE APLICA`).

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)
DIMS   = INTERM / "dims"                       # saída: uma pasta só para as dimensões
DIMS.mkdir(parents=True, exist_ok=True)

# Na execução final encadeada (ETL-26), ambas apontam para 06_limpo.parquet
ENTRADA_LOCALIDADE = INTERM / "03_texto.parquet"             # saída da ETL-14: municípios e regiões limpos
ENTRADA_VIA        = INTERM / "05_nulos_categorias.parquet"  # saída da ETL-16: tipo_via e local_via tratados

COLUNAS_LOCALIDADE = ["cod_ibge", "municipio", "regiao_administrativa"]
COLUNAS_VIA        = ["tipo_via", "local_via"]

## 2. Carregar a entrada

In [2]:
# Lê apenas as colunas necessárias para cada dimensão + id_pessoa para conferir alinhamento
df_loc = pd.read_parquet(ENTRADA_LOCALIDADE, columns=["id_pessoa", *COLUNAS_LOCALIDADE])
df_via = pd.read_parquet(ENTRADA_VIA, columns=["id_pessoa", *COLUNAS_VIA])

# Garante id_pessoa e cod_ibge como inteiros mesmo se a entrada tiver sido salva como texto
df_loc["id_pessoa"] = pd.to_numeric(df_loc["id_pessoa"], errors="raise").astype("int64")
df_loc["cod_ibge"]  = pd.to_numeric(df_loc["cod_ibge"], errors="raise").astype("int64")
df_via["id_pessoa"] = pd.to_numeric(df_via["id_pessoa"], errors="raise").astype("int64")

assert len(df_loc) == len(df_via), "As entradas de localidade e via têm quantidades diferentes de linhas!"
assert df_loc["id_pessoa"].is_unique and df_via["id_pessoa"].is_unique, "id_pessoa duplicado na entrada!"
assert df_loc["id_pessoa"].isin(df_via["id_pessoa"]).all(), "id_pessoa divergente entre as duas entradas!"

print("localidade:", df_loc.shape, "| via:", df_via.shape)
print("mesmas", len(df_loc), "pessoas alinhadas entre as duas entradas")

localidade: (806977, 4) | via: (806977, 3)
mesmas 806977 pessoas alinhadas entre as duas entradas


## 3. Diagnóstico ANTES

In [3]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_nulo": (df.isna().mean() * 100).round(2),
        "n_distintos": df.nunique(),
    })

antes = pd.concat([
    diagnostico(df_loc[COLUNAS_LOCALIDADE]),
    diagnostico(df_via[COLUNAS_VIA]),
])
antes

,dtype,pct_nulo,n_distintos
cod_ibge,int64,0.0,645
municipio,str,0.0,645
regiao_administrativa,str,0.0,16
tipo_via,str,0.0,3
local_via,str,0.0,3


## 4. `dim_localidade`

Cada linha representa um município paulista identificado por sua chave natural `cod_ibge` (7 dígitos) e sua respectiva `regiao_administrativa`.

**Regras aplicadas:**
- Valida relação 1:1 entre `cod_ibge` e `municipio`, além de 1 única `regiao_administrativa` por município.
- Confere ausência de `D OESTE` / `D ALHO` sem apóstrofo (garantindo a limpeza da ETL-14).
- Ordena por `cod_ibge` crescente antes de numerar `sk_localidade` (`1..645`).
- Inclui os membros especiais `-1` (`NÃO INFORMADO`) e `-2` (`NÃO SE APLICA`) com `cod_ibge = <NA>` (`Int64`), compatível com `cod_ibge INTEGER NULL UNIQUE` no DDL (`etl_star_schema.sql`).

In [4]:
def criar_dim_localidade(df: pd.DataFrame) -> pd.DataFrame:
    """Cria a dim_localidade ordenada por cod_ibge com membros especiais -1 e -2."""
    assert df[COLUNAS_LOCALIDADE].isna().sum().sum() == 0, "Há nulos nas colunas de localidade!"

    sem_apostrofo = df["municipio"].str.contains(r"\bD (?:OESTE|ALHO)\b", regex=True, na=False)
    assert not sem_apostrofo.any(), "Município sem apóstrofo detectado (verifique saída da ETL-14)!"

    mun_por_ibge = df.groupby("cod_ibge")["municipio"].nunique()
    reg_por_ibge = df.groupby("cod_ibge")["regiao_administrativa"].nunique()
    ibge_por_mun = df.groupby("municipio")["cod_ibge"].nunique()
    assert (mun_por_ibge == 1).all(), "Há cod_ibge associado a mais de um município!"
    assert (reg_por_ibge == 1).all(), "Há cod_ibge associado a mais de uma região administrativa!"
    assert (ibge_por_mun == 1).all(), "Há município associado a mais de um cod_ibge!"

    dim = (
        df[COLUNAS_LOCALIDADE]
        .drop_duplicates()
        .sort_values("cod_ibge", ascending=True)
        .reset_index(drop=True)
    )
    assert dim["cod_ibge"].is_unique and dim["municipio"].is_unique
    assert dim["cod_ibge"].is_monotonic_increasing, "dim_localidade não está ordenada por cod_ibge!"

    dim.insert(0, "sk_localidade", range(1, len(dim) + 1))

    especiais = pd.DataFrame({
        "sk_localidade": [-1, -2],
        "cod_ibge": pd.array([pd.NA, pd.NA], dtype="Int64"),
        "municipio": ["NÃO INFORMADO", "NÃO SE APLICA"],
        "regiao_administrativa": ["NÃO INFORMADO", "NÃO SE APLICA"],
    })
    dim["cod_ibge"] = dim["cod_ibge"].astype("Int64")
    out = pd.concat([especiais, dim], ignore_index=True)
    out["sk_localidade"] = out["sk_localidade"].astype("int64")
    return out


dim_localidade = criar_dim_localidade(df_loc)
print("dim_localidade shape:", dim_localidade.shape)
dim_localidade.head(10)

dim_localidade shape: (647, 4)


,sk_localidade,cod_ibge,municipio,regiao_administrativa
0,-1,<NA>,NÃO INFORMADO,NÃO INFORMADO
1,-2,<NA>,NÃO SE APLICA,NÃO SE APLICA
2,1,3500105,ADAMANTINA,PRESIDENTE PRUDENTE
3,2,3500204,ADOLFO,SÃO JOSÉ DO RIO PRETO
4,3,3500303,AGUAI,CAMPINAS
5,4,3500402,AGUAS DA PRATA,CAMPINAS
6,5,3500501,AGUAS DE LINDOIA,CAMPINAS
7,6,3500550,AGUAS DE SANTA BARBARA,SOROCABA
8,7,3500600,AGUAS DE SAO PEDRO,CAMPINAS
9,8,3500709,AGUDOS,BAURU


## 5. `dim_via`

Dimensão de combinações distintas de `tipo_via` e `local_via`.

**Regras aplicadas:**
- Valida que `tipo_via` e `local_via` já passaram pelo tratamento de nulos e `'NAO DISPONIVEL'` da ETL-16 (zero `NaN` e zero `'NAO DISPONIVEL'`).
- A combinação em que **ambas** as colunas são `'NÃO INFORMADO'` (`('NÃO INFORMADO', 'NÃO INFORMADO')`) não recebe `sk_via > 0`, sendo absorvida pelo membro especial **`-1`** para respeitar `UNIQUE (tipo_via, local_via)` do DDL.
- Combinações parciais (onde apenas uma coluna é `'NÃO INFORMADO'`, como `('VIAS URBANAS', 'NÃO INFORMADO')`) recebem `sk_via > 0` para preservar a informação conhecida.
- Ordena por `['tipo_via', 'local_via']` antes de numerar `sk_via` (`1..8`).

In [5]:
def criar_dim_via(df: pd.DataFrame) -> pd.DataFrame:
    """Cria a dim_via com uma linha por combinação válida (tipo_via, local_via) e membros -1 e -2."""
    assert df[COLUNAS_VIA].isna().sum().sum() == 0, "Há nulos em tipo_via/local_via (verifique ETL-16)!"
    for col in COLUNAS_VIA:
        assert not (df[col] == "NAO DISPONIVEL").any(), f"'{col}' ainda contém 'NAO DISPONIVEL' (verifique ETL-16)!"

    dim = (
        df[COLUNAS_VIA]
        .drop_duplicates()
        .sort_values(COLUNAS_VIA, ascending=True)
        .reset_index(drop=True)
    )

    # Combinações totalmente desconhecidas ou não aplicáveis ficam nos membros -1 e -2
    so_nao_informado = (dim[COLUNAS_VIA] == "NÃO INFORMADO").all(axis=1)
    so_nao_se_aplica = (dim[COLUNAS_VIA] == "NÃO SE APLICA").all(axis=1)
    dim = dim[~(so_nao_informado | so_nao_se_aplica)].reset_index(drop=True)

    assert not dim.duplicated(subset=COLUNAS_VIA).any()
    assert dim[COLUNAS_VIA].apply(tuple, axis=1).is_monotonic_increasing, "dim_via não está ordenada!"
    dim.insert(0, "sk_via", range(1, len(dim) + 1))

    especiais = pd.DataFrame({
        "sk_via": [-1, -2],
        "tipo_via": ["NÃO INFORMADO", "NÃO SE APLICA"],
        "local_via": ["NÃO INFORMADO", "NÃO SE APLICA"],
    })
    out = pd.concat([especiais, dim], ignore_index=True)
    out["sk_via"] = out["sk_via"].astype("int64")
    return out


dim_via = criar_dim_via(df_via)
print("dim_via shape:", dim_via.shape)
dim_via

dim_via shape: (10, 3)


,sk_via,tipo_via,local_via
0,-1,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA
2,1,ESTRADAS E RODOVIAS,NÃO INFORMADO
3,2,ESTRADAS E RODOVIAS,PRIVADO
4,3,ESTRADAS E RODOVIAS,PUBLICO
5,4,NÃO INFORMADO,PRIVADO
6,5,NÃO INFORMADO,PUBLICO
7,6,VIAS URBANAS,NÃO INFORMADO
8,7,VIAS URBANAS,PRIVADO
9,8,VIAS URBANAS,PUBLICO


## 6. Conferir, validar cobertura na base e salvar

Além das validações de unicidade, ordenação e limites de tamanho do DDL (`etl_star_schema.sql`), esta seção define as funções `atribuir_sk_localidade` e `atribuir_sk_via` para reutilização na construção da tabela fato (`ETL-23`).

In [6]:
def atribuir_sk_localidade(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Retorna a sk_localidade de cada linha a partir de cod_ibge (-1 se ausente). Reutilizável na ETL-23."""
    reais = dim[dim["sk_localidade"] > 0].copy()
    mapa = reais.set_index(reais["cod_ibge"].astype("int64"))["sk_localidade"]
    codigos = pd.to_numeric(df["cod_ibge"], errors="coerce")
    return codigos.map(mapa).fillna(-1).astype("int64")


def atribuir_sk_via(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Retorna a sk_via de cada linha a partir do par (tipo_via, local_via) (-1 se ausente). Reutilizável na ETL-23."""
    chaves = df[COLUNAS_VIA].fillna("NÃO INFORMADO").replace("NAO DISPONIVEL", "NÃO INFORMADO")
    m = chaves.merge(dim, on=COLUNAS_VIA, how="left")
    return pd.Series(m["sk_via"].fillna(-1).astype("int64").to_numpy(), index=df.index)


validacoes = [
    ("dim_localidade", dim_localidade, "sk_localidade", ["cod_ibge"], {"municipio": 60, "regiao_administrativa": 40}),
    ("dim_via", dim_via, "sk_via", COLUNAS_VIA, {"tipo_via": 30, "local_via": 20}),
]

for nome, dim, sk, chave_natural, limites in validacoes:
    reais = dim[dim[sk] > 0]
    assert dim[sk].is_unique, f"{nome}: {sk} repetida!"
    assert list(reais[sk]) == list(range(1, len(reais) + 1)), f"{nome}: {sk} não sequencial!"
    assert set(dim.loc[dim[sk] < 0, sk]) == {-1, -2}, f"{nome}: membros -1 e -2 ausentes!"
    assert not reais.duplicated(subset=chave_natural).any(), f"{nome}: chave natural duplicada nos reais!"
    if nome == "dim_via":
        assert not dim.duplicated(subset=chave_natural).any(), "dim_via: combinação duplicada considerando especiais!"
    else:
        assert dim["municipio"].is_unique, "dim_localidade: municipio duplicado!"
        assert dim.loc[dim[sk] < 0, "cod_ibge"].isna().all(), "dim_localidade: especiais devem ter cod_ibge nulo!"
    for col, lim in limites.items():
        max_len = int(dim[col].str.len().max())
        assert max_len <= lim, f"{nome}.{col} excede VARCHAR({lim}): {max_len}"
    dim.to_parquet(DIMS / f"{nome}.parquet", index=False)
    print(f"{nome} | {len(dim)} linhas | {sk} única: True | duplicatas chave natural: 0 | salva em {DIMS / f'{nome}.parquet'}")

dim_localidade | 647 linhas | sk_localidade única: True | duplicatas chave natural: 0 | salva em C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\dims\dim_localidade.parquet
dim_via | 10 linhas | sk_via única: True | duplicatas chave natural: 0 | salva em C:\Users\gcnt1\OneDrive\Área de Trabalho\Pastas Diversas\Integracao_BD\data\intermediarios\dims\dim_via.parquet


In [7]:
# Confere que todas as 806.977 pessoas encontram chave válida nas duas dimensões
df_loc["sk_localidade"] = atribuir_sk_localidade(df_loc, dim_localidade)
df_via["sk_via"] = atribuir_sk_via(df_via, dim_via)

cont_loc = df_loc["sk_localidade"].value_counts()
cont_via = df_via["sk_via"].value_counts()

assert cont_loc.sum() == len(df_loc) and (df_loc["sk_localidade"] > 0).all()
assert cont_via.sum() == len(df_via) and df_via["sk_via"].isin(dim_via["sk_via"]).all()

print("pessoas mapeadas | dim_localidade:", int(cont_loc.sum()), "| dim_via:", int(cont_via.sum()))

depois = pd.DataFrame({
    "linhas_dimensao": [len(dim_localidade), len(dim_via)],
    "membros_reais": [int((dim_localidade["sk_localidade"] > 0).sum()), int((dim_via["sk_via"] > 0).sum())],
    "pessoas_-1 (NÃO INFORMADO)": [int(cont_loc.get(-1, 0)), int(cont_via.get(-1, 0))],
    "pessoas_-2 (NÃO SE APLICA)": [int(cont_loc.get(-2, 0)), int(cont_via.get(-2, 0))],
}, index=["dim_localidade", "dim_via"])
depois

pessoas mapeadas | dim_localidade: 806977 | dim_via: 806977


,linhas_dimensao,membros_reais,pessoas_-1 (NÃO INFORMADO),pessoas_-2 (NÃO SE APLICA)
dim_localidade,647,645,0,0
dim_via,10,8,663,0


In [8]:
# Distribuição da dim_via com quantidade de pessoas por combinação
dim_via.assign(pessoas=dim_via["sk_via"].map(cont_via).fillna(0).astype("int64"))

,sk_via,tipo_via,local_via,pessoas
0,-1,NÃO INFORMADO,NÃO INFORMADO,663
1,-2,NÃO SE APLICA,NÃO SE APLICA,0
2,1,ESTRADAS E RODOVIAS,NÃO INFORMADO,19757
3,2,ESTRADAS E RODOVIAS,PRIVADO,150
4,3,ESTRADAS E RODOVIAS,PUBLICO,113686
5,4,NÃO INFORMADO,PRIVADO,7
6,5,NÃO INFORMADO,PUBLICO,303
7,6,VIAS URBANAS,NÃO INFORMADO,107156
8,7,VIAS URBANAS,PRIVADO,4218
9,8,VIAS URBANAS,PUBLICO,561037


In [9]:
# Top 10 municípios com mais registros na dim_localidade
dim_localidade.assign(
    pessoas=dim_localidade["sk_localidade"].map(cont_loc).fillna(0).astype("int64")
).sort_values("pessoas", ascending=False).head(10)

,sk_localidade,cod_ibge,municipio,regiao_administrativa,pessoas
564,563,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,186723
489,488,3543402,RIBEIRAO PRETO,RIBEIRÃO PRETO,21395
110,109,3509502,CAMPINAS,CAMPINAS,20429
214,213,3518800,GUARULHOS,METROPOLITANA DE SÃO PAULO,19452
558,557,3549805,SAO JOSE DO RIO PRETO,SÃO JOSÉ DO RIO PRETO,16521
583,582,3552205,SOROCABA,SOROCABA,14786
546,545,3548708,SAO BERNARDO DO CAMPO,METROPOLITANA DE SÃO PAULO,14494
536,535,3547809,SANTO ANDRE,METROPOLITANA DE SÃO PAULO,11767
295,294,3525904,JUNDIAI,CAMPINAS,11090
559,558,3549904,SAO JOSE DOS CAMPOS,SÃO JOSÉ DOS CAMPOS,10956


## 7. Resumo

**O que foi feito:** criação das dimensões `dim_localidade` e `dim_via` do modelo estrela, com chaves substitutas sequenciais (`sk_*`), ordenação determinística e membros especiais `-1` (`NÃO INFORMADO`) e `-2` (`NÃO SE APLICA`). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Entradas:** `03_texto.parquet` (`ETL-14`) para `dim_localidade` (municípios com apóstrofos `D'OESTE`/`D'ALHO` restaurados) e `05_nulos_categorias.parquet` (`ETL-16`) para `dim_via` (`NAO DISPONIVEL` e nulos já convertidos para `NÃO INFORMADO`). Na execução final encadeada (`ETL-26`), ambas passam a ler `06_limpo.parquet`.

**Resultados:**
- `dim_localidade`: **647 linhas** (`645` municípios ordenados por `cod_ibge` + membros `-1` e `-2` com `cod_ibge` nulo em `Int64`).
- `dim_via`: **10 linhas** (`8` combinações reais ordenadas por `(tipo_via, local_via)` + membros `-1` e `-2`).

**Decisão sobre `dim_via` (`('NÃO INFORMADO', 'NÃO INFORMADO')`):** nos dados existem 9 combinações de `(tipo_via, local_via)`, mas a combinação dupla `('NÃO INFORMADO', 'NÃO INFORMADO')` (`663` pessoas) é direcionada ao membro `-1` para preservar a restrição `UNIQUE (tipo_via, local_via)` do DDL (`etl_star_schema.sql`). Por isso a dimensão tem `10` linhas (em vez de `11` estimadas inicialmente no modelo lógico).

**O que foi conferido:**
- relação 1:1 entre `cod_ibge` e `municipio`, e 1 única `regiao_administrativa` por município;
- chaves `sk_localidade` e `sk_via` únicas e sequenciais, sem duplicatas nas chaves naturais e dentro dos limites de `VARCHAR` do DDL;
- todas as `806.977` pessoas da base recebem chave válida nas duas dimensões (`0` sem chave).

**Para a ETL-23 (tabela fato):** o notebook disponibiliza as funções `atribuir_sk_localidade` e `atribuir_sk_via` para mapear cada linha da base limpa às chaves das duas dimensões.